# Chiral Tensor Mode Analysis

## Result: NO parity-odd tensor term exists in the minimal EC theory

The analysis in `02_parity_odd_source_terms.md` and `03_chiral_mode_equations.md`
establishes that the minimal Einstein-Cartan theory on FRW produces:

$$v_{k,L}'' + (k^2 - a''/a)\,v_{k,L} = 0$$
$$v_{k,R}'' + (k^2 - a''/a)\,v_{k,R} = 0$$

**Identical equations.** No chirality splitting. $\Delta\chi = 0$ exactly.

---

Since the real model has no parity-odd tensor term, this notebook:
1. Confirms $\Delta\chi = 0$ numerically for the actual model
2. Shows what chirality WOULD look like with a toy Chern-Simons term,
   to illustrate the contrast with the actual null result

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt

# Parameters (same as tensor_spectrum notebook)
rho_crit = 0.21  # M_Pl^4
alpha2 = 8 * np.pi * rho_crit / 3
alpha = np.sqrt(alpha2)
a_b = 1.0
k_b = a_b * np.sqrt(2) * alpha

def a_cosmic(t):
    return a_b * (1 + 4 * alpha2 * t**2)**0.25

def app_over_a_cosmic(t):
    return 2 * alpha2 * a_b**2 / (1 + 4 * alpha2 * t**2)**1.5

# Build conformal time grid
N_t = 100000
t_grid = np.linspace(-20/alpha, 20/alpha, N_t)
eta_grid = np.zeros(N_t)
for i in range(1, N_t):
    dt = t_grid[i] - t_grid[i-1]
    eta_grid[i] = eta_grid[i-1] + 0.5 * dt * (1/a_cosmic(t_grid[i-1]) + 1/a_cosmic(t_grid[i]))

U_grid = np.array([app_over_a_cosmic(t) for t in t_grid])
U_interp = interp1d(eta_grid, U_grid, kind='cubic', fill_value=0.0, bounds_error=False)

print(f"k_b = {k_b:.4f}")
print(f"Conformal time range: [{eta_grid[0]:.4f}, {eta_grid[-1]:.4f}]")

## 1. Confirm Δχ = 0 for the Actual Model

Solve left and right modes with the SAME equation (μ = 0) and verify.

In [ ]:
def solve_chiral_mode(k_val, mu_func=None):
    """Solve v'' + (k^2 - U(eta) + s*mu(eta)*k) v = 0
    for s = +1 (left) and s = -1 (right).
    
    If mu_func is None, both polarizations are identical.
    Returns (beta_L_sq, beta_R_sq, chirality).
    """
    eta_start = eta_grid[0]
    eta_end = eta_grid[-1]
    
    results = {}
    for s, label in [(+1, 'L'), (-1, 'R')]:
        norm = 1.0 / np.sqrt(2 * k_val)
        v0_re = norm * np.cos(-k_val * eta_start)
        v0_im = norm * np.sin(-k_val * eta_start)
        vp0_re = norm * k_val * np.sin(-k_val * eta_start)
        vp0_im = -norm * k_val * np.cos(-k_val * eta_start)
        
        def rhs(eta, y):
            v_re, v_im, vp_re, vp_im = y
            U_val = float(U_interp(eta))
            mu_val = mu_func(eta) if mu_func is not None else 0.0
            omega2 = k_val**2 - U_val + s * mu_val * k_val
            return [vp_re, vp_im, -omega2 * v_re, -omega2 * v_im]
        
        sol = solve_ivp(rhs, [eta_start, eta_end],
                       [v0_re, v0_im, vp0_re, vp0_im],
                       rtol=1e-10, atol=1e-12, method='DOP853',
                       dense_output=True)
        
        # Extract Bogoliubov at tail
        n_pts = 100
        eta_tail = np.linspace(0.8 * eta_end, eta_end, n_pts)
        y_tail = sol.sol(eta_tail)
        v = y_tail[0] + 1j * y_tail[1]
        vp = y_tail[2] + 1j * y_tail[3]
        
        norm2 = np.sqrt(2 * k_val) / 2
        beta = norm2 * (v + 1j * vp / k_val) * np.exp(-1j * k_val * eta_tail)
        results[label] = np.mean(np.abs(beta)**2)
    
    bL = results['L']
    bR = results['R']
    chi = (bL - bR) / (bL + bR) if (bL + bR) > 0 else 0.0
    return bL, bR, chi

# Test with mu = 0 (actual EC model)
k_test = 0.3 * k_b
bL, bR, chi = solve_chiral_mode(k_test, mu_func=None)
print(f"Actual EC model (mu = 0):")
print(f"  k/k_b = {k_test/k_b:.3f}")
print(f"  |beta_L|^2 = {bL:.6f}")
print(f"  |beta_R|^2 = {bR:.6f}")
print(f"  Chirality  = {chi:.2e}")
print(f"")
print(f"  Δχ = 0 to machine precision: {'YES' if abs(chi) < 1e-10 else 'NO'}")

In [ ]:
# Confirm for multiple k values
print(f"{'k/k_b':>8}  {'|beta_L|^2':>12}  {'|beta_R|^2':>12}  {'Δχ':>12}")
print("-" * 52)

k_tests = [0.1, 0.2, 0.5, 1.0, 2.0]
for kr in k_tests:
    k = kr * k_b
    bL, bR, chi = solve_chiral_mode(k, mu_func=None)
    print(f"{kr:8.2f}  {bL:12.6f}  {bR:12.6f}  {chi:12.2e}")

print(f"\nAll chiralities are zero to machine precision.")
print(f"This confirms: NO parity violation in the minimal EC bounce.")

## 2. Toy Comparison: What Chirality WOULD Look Like

For illustration, add a toy Chern-Simons-like term:

$$\mu(\eta) = \mu_0 \cdot U(\eta) / U_0$$

This is NOT present in the actual model. It shows the contrast
between the actual null result and a hypothetical parity-violating model.

In [ ]:
# Toy parity-odd term (NOT in EC theory)
U_max = 2 * alpha2 * a_b**2

def mu_toy(eta, strength=0.5):
    """Toy Chern-Simons-like term, localized at the bounce."""
    return strength * float(U_interp(eta)) / U_max * k_b

# Solve with toy parity violation for several strengths
strengths = [0.0, 0.01, 0.05, 0.1, 0.3, 0.5]
k_test = 0.3 * k_b

print(f"Toy Chern-Simons at k/k_b = {k_test/k_b:.2f}:")
print(f"{'μ₀':>8}  {'|β_L|²':>10}  {'|β_R|²':>10}  {'Δχ':>10}")
print("-" * 45)

for s in strengths:
    mu_fn = lambda eta, s=s: mu_toy(eta, strength=s)
    bL, bR, chi = solve_chiral_mode(k_test, mu_func=mu_fn)
    print(f"{s:8.3f}  {bL:10.4f}  {bR:10.4f}  {chi:10.4f}")

print(f"\nNote: μ₀ = 0 is the ACTUAL EC model (no chirality).")
print(f"All μ₀ > 0 are TOY models showing hypothetical parity violation.")

In [ ]:
# Chirality spectrum for toy model
k_values = np.logspace(-1, 0.8, 25) * k_b
mu_strength = 0.3  # Moderate toy parity violation

chi_actual = []
chi_toy = []

for k in k_values:
    # Actual model
    _, _, c0 = solve_chiral_mode(k, mu_func=None)
    chi_actual.append(c0)
    
    # Toy model
    mu_fn = lambda eta: mu_toy(eta, strength=mu_strength)
    _, _, ct = solve_chiral_mode(k, mu_func=mu_fn)
    chi_toy.append(ct)

chi_actual = np.array(chi_actual)
chi_toy = np.array(chi_toy)

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogx(k_values/k_b, chi_actual, 'b.-', label='Actual EC model (μ=0)', markersize=8)
ax.semilogx(k_values/k_b, chi_toy, 'r.-', label=f'Toy CS model (μ₀={mu_strength})', markersize=8)
ax.axhline(0, color='black', ls='-', alpha=0.3)
ax.set_xlabel(r'$k / k_b$')
ax.set_ylabel(r'Chirality $\Delta\chi = (P_L - P_R)/(P_L + P_R)$')
ax.set_title('Tensor Chirality: Actual EC vs Toy Parity-Violating Model')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.5, 0.5)
plt.tight_layout()
plt.savefig('chirality_comparison.png', dpi=150)
plt.show()

print(f"\nMax |Δχ| in actual EC model: {np.max(np.abs(chi_actual)):.2e}")
print(f"Max |Δχ| in toy CS model:     {np.max(np.abs(chi_toy)):.4f}")

In [ ]:
print("=" * 60)
print("CHIRAL TENSOR ANALYSIS: SUMMARY")
print("=" * 60)
print()
print("Actual Einstein-Cartan model:")
print(f"  Parity-odd tensor source:  NONE")
print(f"  Chirality Δχ:              0 (exact)")
print(f"  Left/right splitting:      NONE")
print()
print("For comparison, toy Chern-Simons model:")
print(f"  Parity-odd tensor source:  μ(η) = μ₀ U(η)/U₀ × k_b")
print(f"  Chirality Δχ:              O(μ₀) ~ 0.01-0.3")
print(f"  Left/right splitting:      YES")
print()
print("CONCLUSION:")
print("  The minimal spin-torsion bounce produces ZERO chirality.")
print("  Chirality requires physics BEYOND Einstein-Cartan gravity.")
print("  The toy model shows what a real signal would look like,")
print("  but this signal is NOT present in the actual theory.")